In [1]:
# Install (run once)
!pip install mysql-connector-python

In [2]:
import requests
import pandas as pd
import mysql.connector

In [3]:
API_KEY = "b3d7f61023msh362a5491e103391p1f5aecjsnb0cb3e72401e" 

HEADERS = {
    "X-RapidAPI-Key": API_KEY,
    "X-RapidAPI-Host": "cricbuzz-cricket.p.rapidapi.com"
}

In [4]:
def get_live_matches():
    url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live"
    
    response = requests.get(url, headers=HEADERS)
    data = response.json()
    
    return data

In [5]:
data = get_live_matches()
print(data)

{'typeMatches': [{'matchType': 'International', 'seriesMatches': [{'seriesAdWrapper': {'seriesId': 12081, 'seriesName': 'United Arab Emirates tour of Nepal 2026', 'matches': [{'matchInfo': {'matchId': 153803, 'seriesId': 12081, 'seriesName': 'United Arab Emirates tour of Nepal 2026', 'matchDesc': '2nd T20I', 'matchFormat': 'T20', 'startDate': '1776770100000', 'endDate': '1776782700000', 'state': 'Complete', 'status': 'Nepal won by 8 wkts', 'team1': {'teamId': 7, 'teamName': 'United Arab Emirates', 'teamSName': 'UAE', 'imageId': 776242}, 'team2': {'teamId': 72, 'teamName': 'Nepal', 'teamSName': 'NEP', 'imageId': 776331}, 'venueInfo': {'id': 458, 'ground': 'Tribhuvan University International Cricket Ground', 'city': 'Kirtipur', 'timezone': '+05:45', 'latitude': '27.6768145', 'longitude': '85.287199'}, 'currBatTeamId': 72, 'seriesStartDt': '1776643200000', 'seriesEndDt': '1776902400000', 'isTimeAnnounced': True, 'stateTitle': 'Complete'}, 'matchScore': {'team1Score': {'inngs1': {'inningsI

In [6]:
def extract_matches(data):
    matches = []

    for type_match in data.get("typeMatches", []):
        for series in type_match.get("seriesMatches", []):
            wrapper = series.get("seriesAdWrapper", {})

            for match in wrapper.get("matches", []):
                info = match.get("matchInfo", {})
                score = match.get("matchScore", {})

                matches.append({
                    # 🔑 Primary Key
                    "match_id": info.get("matchId"),

                    # 📊 Series Info
                    "series_id": info.get("seriesId"),
                    "series_name": info.get("seriesName"),

                    # 🏏 Match Info
                    "match_desc": info.get("matchDesc"),
                    "match_format": info.get("matchFormat"),
                    "status": info.get("status"),

                    # 👥 Teams
                    "team1": info.get("team1", {}).get("teamName"),
                    "team2": info.get("team2", {}).get("teamName"),

                    # 📍 Venue
                    "venue": info.get("venueInfo", {}).get("ground"),
                    "city": info.get("venueInfo", {}).get("city"),

                    # 🕒 Dates (converted from timestamp → readable)
                    "start_date": info.get("startDate"),
                    "end_date": info.get("endDate"),

                    # ⚡ Optional Score Summary (safe extraction)
                    "team1_runs": score.get("team1Score", {}).get("inngs1", {}).get("runs"),
                    "team2_runs": score.get("team2Score", {}).get("inngs1", {}).get("runs")
                })

    return matches

In [7]:
matches = extract_matches(data)
df = pd.DataFrame(matches)
df

,match_id,series_id,series_name,match_desc,match_format,status,team1,team2,venue,city,start_date,end_date,team1_runs,team2_runs
0,153803,12081,United Arab Emirates tour of Nepal 2026,2nd T20I,T20,Nepal won by 8 wkts,United Arab Emirates,Nepal,Tribhuvan University International Cricket Ground,Kirtipur,1776770100000,1776782700000,128,129.0
1,151856,9241,Indian Premier League 2026,31st Match,T20,Delhi Capitals opt to bowl,Sunrisers Hyderabad,Delhi Capitals,Rajiv Gandhi International Stadium,Hyderabad,1776780000000,1776792600000,152,NaN
2,149260,11537,Pakistan Super League 2026,31st Match,T20,Rawalpindiz opt to bat,Rawalpindiz,Multan Sultans,National Stadium,Karachi,1776780000000,1776792600000,103,NaN
3,149249,11537,Pakistan Super League 2026,30th Match,T20,Lahore Qalandars won by 9 runs,Lahore Qalandars,Quetta Gladiators,Gaddafi Stadium,Lahore,1776763800000,1776776400000,197,188.0
4,153439,12015,ICC Womens T20I Challenge Trophy 2026,6th Match,T20,Italy Women won by 4 wkts,Vanuatu Women,Italy Women,Gahanga International Cricket Stadium,Kigali City,1776769200000,1776781800000,114,115.0
5,153428,12015,ICC Womens T20I Challenge Trophy 2026,5th Match,T20,Rwanda Women won by 1 run,Rwanda Women,United States of America Women,Gahanga International Cricket Stadium,Kigali City,1776754800000,1776767400000,127,126.0


In [27]:
df.columns

Index(['match_id', 'series_id', 'series_name', 'match_desc', 'match_format',
       'status', 'team1', 'team2', 'venue', 'city', 'start_date', 'end_date',
       'team1_runs', 'team2_runs'],
      dtype='object')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   match_id      6 non-null      int64  
 1   series_id     6 non-null      int64  
 2   series_name   6 non-null      object 
 3   match_desc    6 non-null      object 
 4   match_format  6 non-null      object 
 5   status        6 non-null      object 
 6   team1         6 non-null      object 
 7   team2         6 non-null      object 
 8   venue         6 non-null      object 
 9   city          6 non-null      object 
 10  start_date    6 non-null      object 
 11  end_date      6 non-null      object 
 12  team1_runs    6 non-null      int64  
 13  team2_runs    4 non-null      float64
dtypes: float64(1), int64(3), object(10)
memory usage: 804.0+ bytes


In [9]:
df["start_date"] = pd.to_datetime(df["start_date"], unit='ms')
df["end_date"] = pd.to_datetime(df["end_date"], unit='ms')

C:\Users\hp\AppData\Local\Temp\ipykernel_20536\892971224.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df["start_date"] = pd.to_datetime(df["start_date"], unit='ms')
C:\Users\hp\AppData\Local\Temp\ipykernel_20536\892971224.py:2: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df["end_date"] = pd.to_datetime(df["end_date"], unit='ms')


In [10]:
df = df.fillna({
    "team2_runs": 0
})

In [11]:
df.head()

,match_id,series_id,series_name,match_desc,match_format,status,team1,team2,venue,city,start_date,end_date,team1_runs,team2_runs
0,153803,12081,United Arab Emirates tour of Nepal 2026,2nd T20I,T20,Nepal won by 8 wkts,United Arab Emirates,Nepal,Tribhuvan University International Cricket Ground,Kirtipur,2026-04-21 11:15:00,2026-04-21 14:45:00,128,129.0
1,151856,9241,Indian Premier League 2026,31st Match,T20,Delhi Capitals opt to bowl,Sunrisers Hyderabad,Delhi Capitals,Rajiv Gandhi International Stadium,Hyderabad,2026-04-21 14:00:00,2026-04-21 17:30:00,152,0.0
2,149260,11537,Pakistan Super League 2026,31st Match,T20,Rawalpindiz opt to bat,Rawalpindiz,Multan Sultans,National Stadium,Karachi,2026-04-21 14:00:00,2026-04-21 17:30:00,103,0.0
3,149249,11537,Pakistan Super League 2026,30th Match,T20,Lahore Qalandars won by 9 runs,Lahore Qalandars,Quetta Gladiators,Gaddafi Stadium,Lahore,2026-04-21 09:30:00,2026-04-21 13:00:00,197,188.0
4,153439,12015,ICC Womens T20I Challenge Trophy 2026,6th Match,T20,Italy Women won by 4 wkts,Vanuatu Women,Italy Women,Gahanga International Cricket Stadium,Kigali City,2026-04-21 11:00:00,2026-04-21 14:30:00,114,115.0


In [12]:
def get_scorecard(match_id):
    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/hscard"
    
    response = requests.get(url, headers=HEADERS)
    return response.json()

In [13]:
scorecard = get_scorecard(df['match_id'][0])
print(scorecard)

{'scorecard': [{'inningsid': 1, 'batsman': [{'id': 18987, 'balls': 1, 'runs': 0, 'fours': 0, 'sixes': 0, 'strkrate': '0', 'name': 'Muhammad Waseem', 'nickname': 'Muhammad Waseem', 'iscaptain': True, 'iskeeper': False, 'outdec': 'lbw b Hemant Dhami', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': ''}, {'id': 14999, 'balls': 22, 'runs': 30, 'fours': 2, 'sixes': 2, 'strkrate': '136.36', 'name': 'Adeeb Usmani', 'nickname': 'Adeeb Usmani', 'iscaptain': False, 'iskeeper': True, 'outdec': 'lbw b Sandeep Lamichhane', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': 'IN'}, {'id': 14227, 'balls': 4, 'runs': 1, 'fours': 0, 'sixes': 0, 'strkrate': '25', 'name': 'Alishan Sharafu', 'nickname

In [14]:
def extract_players(scorecard, match_id):
    players = []

    for innings in scorecard.get("scorecard", []):
        innings_id = innings.get("inningsid")
        batsmen = innings.get("batsman", [])

        for b in batsmen:
            players.append({
                "player_id": b.get("id"),
                "match_id": match_id,
                "innings_id": innings_id,

                "name": b.get("name"),
                "runs": b.get("runs", 0),
                "balls": b.get("balls", 0),
                "fours": b.get("fours", 0),
                "sixes": b.get("sixes", 0),

                # convert strike rate safely (string → float)
                "strike_rate": float(b.get("strkrate", 0)) if b.get("strkrate") else 0.0,

                # handle empty dismissal
                "out_desc": b.get("outdec") if b.get("outdec") else "not out"
            })

    return players

In [15]:
players = extract_players(scorecard, match_id=142142)

import pandas as pd
df_players = pd.DataFrame(players)

print(df_players.head())

   player_id  match_id  innings_id                   name  runs  balls  fours  \
0      18987    142142           1        Muhammad Waseem     0      1      0   
1      14999    142142           1           Adeeb Usmani    30     22      2   
2      14227    142142           1        Alishan Sharafu     1      4      0   
3       6322    142142           1  Harpreet Singh Bhatia     1      3      0   
4       8351    142142           1          Akshdeep Nath    53     42      3   

   sixes  strike_rate                     out_desc  
0      0         0.00           lbw b Hemant Dhami  
1      2       136.36     lbw b Sandeep Lamichhane  
2      0        25.00  c Lokesh Bam b Nandan Yadav  
3      0        33.33           lbw b Hemant Dhami  
4      4       126.19                      not out  


In [28]:
df_players.columns

Index(['player_id', 'match_id', 'innings_id', 'name', 'runs', 'balls', 'fours',
       'sixes', 'strike_rate', 'out_desc'],
      dtype='object')

In [16]:
df_players

,player_id,match_id,innings_id,name,runs,balls,fours,sixes,strike_rate,out_desc
0,18987,142142,1,Muhammad Waseem,0,1,0,0,0.00,lbw b Hemant Dhami
1,14999,142142,1,Adeeb Usmani,30,22,2,2,136.36,lbw b Sandeep Lamichhane
2,14227,142142,1,Alishan Sharafu,1,4,0,0,25.00,c Lokesh Bam b Nandan Yadav
3,6322,142142,1,Harpreet Singh Bhatia,1,3,0,0,33.33,lbw b Hemant Dhami
4,8351,142142,1,Akshdeep Nath,53,42,3,4,126.19,not out
5,1464054,142142,1,Sohaib Khan,0,5,0,0,0.00,c Hemant Dhami b Shahab Alam
6,22841,142142,1,Nilansh Keswani,0,2,0,0,0.00,run out (Dipendra Singh Airee/Lokesh Bam)
7,1448301,142142,1,Muhammad Arfan,32,36,2,1,88.89,c Dipendra Singh Airee b Sandeep Lamichhane
8,1447465,142142,1,Khuzaima Tanveer,0,1,0,0,0.00,c Dipendra Singh Airee b Sandeep Lamichhane
9,51658,142142,1,Muhammad Zuhaib,8,5,1,0,160.00,not out


In [17]:
df.drop_duplicates(inplace=True)
df_players.dropna(inplace=True)

In [18]:
def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="Abc@12345",
        database="cricket_db"
    )

In [19]:
conn = get_connection()
cur = conn.cursor()


cur.execute("""
CREATE TABLE IF NOT EXISTS matches (
    match_id BIGINT PRIMARY KEY,

    --  Series Info
    series_id BIGINT,
    series_name VARCHAR(255),

    --  Match Info
    match_desc VARCHAR(100),
    match_format VARCHAR(20),
    status TEXT,

    --  Teams
    team1 VARCHAR(100),
    team2 VARCHAR(100),

    --  Venue
    venue VARCHAR(100),
    city VARCHAR(100),

    --  Dates
    start_date DATETIME,
    end_date DATETIME,

    --  Optional Score Summary
    team1_runs INT,
    team2_runs INT
)
""")

In [20]:

# players table with FOREIGN KEY
cur.execute("""
CREATE TABLE IF NOT EXISTS players (
    player_id BIGINT,
    match_id BIGINT,
    name VARCHAR(100),
    runs INT,
    balls INT,
    fours INT,
    sixes INT,
    strike_rate FLOAT,
    out_desc TEXT,

    PRIMARY KEY (player_id, match_id),

    FOREIGN KEY (match_id) REFERENCES matches(match_id)
    ON DELETE CASCADE
)
""")

conn.commit()
cur.close()
conn.close()

In [21]:
def extract_matches(data):
    matches = []

    for type_match in data.get("typeMatches", []):
        for series in type_match.get("seriesMatches", []):
            wrapper = series.get("seriesAdWrapper", {})

            for match in wrapper.get("matches", []):
                info = match.get("matchInfo", {})
                score = match.get("matchScore", {})

                matches.append({
                    "match_id": info.get("matchId"),
                    "series_id": info.get("seriesId"),
                    "series_name": info.get("seriesName"),
                    "match_desc": info.get("matchDesc"),
                    "match_format": info.get("matchFormat"),
                    "status": info.get("status"),
                    "team1": info.get("team1", {}).get("teamName"),
                    "team2": info.get("team2", {}).get("teamName"),
                    "venue": info.get("venueInfo", {}).get("ground"),
                    "city": info.get("venueInfo", {}).get("city"),
                    "start_date": info.get("startDate"),
                    "end_date": info.get("endDate"),
                    "team1_runs": score.get("team1Score", {}).get("inngs1", {}).get("runs", 0),
                    "team2_runs": score.get("team2Score", {}).get("inngs1", {}).get("runs", 0)
                })

    return matches

In [22]:
def convert_dates(matches):
    for m in matches:
        m["start_date"] = pd.to_datetime(m["start_date"], unit='ms') if m["start_date"] else None
        m["end_date"] = pd.to_datetime(m["end_date"], unit='ms') if m["end_date"] else None
    return matches

In [23]:
def insert_matches(matches):
    conn = get_connection()
    cur = conn.cursor()

    query = """
    INSERT INTO matches (
        match_id, series_id, series_name, match_desc, match_format,
        status, team1, team2, venue, city,
        start_date, end_date, team1_runs, team2_runs
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE 
        series_name=VALUES(series_name),
        match_desc=VALUES(match_desc),
        match_format=VALUES(match_format),
        status=VALUES(status),
        team1=VALUES(team1),
        team2=VALUES(team2),
        venue=VALUES(venue),
        city=VALUES(city),
        start_date=VALUES(start_date),
        end_date=VALUES(end_date),
        team1_runs=VALUES(team1_runs),
        team2_runs=VALUES(team2_runs)
    """

    values = []
    for m in matches:
        values.append((
            m.get("match_id"),
            m.get("series_id"),
            m.get("series_name"),
            m.get("match_desc"),
            m.get("match_format"),
            m.get("status"),
            m.get("team1"),
            m.get("team2"),
            m.get("venue"),
            m.get("city"),
            m.get("start_date"),
            m.get("end_date"),
            m.get("team1_runs") or 0,
            m.get("team2_runs") or 0
        ))

    cur.executemany(query, values)
    conn.commit()
    cur.close()
    conn.close()

In [24]:
def extract_players(scorecard, match_id):
    players = []

    for innings in scorecard.get("scorecard", []):
        for b in innings.get("batsman", []):
            players.append({
                "player_id": b.get("id"),
                "match_id": match_id,
                "name": b.get("name"),
                "runs": b.get("runs", 0),
                "balls": b.get("balls", 0),
                "fours": b.get("fours", 0),
                "sixes": b.get("sixes", 0),
                "strike_rate": float(b.get("strkrate", 0) or 0),
                "out_desc": b.get("outdec") or "not out"
            })

    return players

In [25]:
def insert_players(players):
    conn = get_connection()
    cur = conn.cursor()

    query = """
    INSERT INTO players 
    (player_id, match_id, name, runs, balls, fours, sixes, strike_rate, out_desc)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    values = []
    for p in players:
        values.append((
            p.get("player_id"),
            p.get("match_id"),
            p.get("name"),
            p.get("runs", 0),
            p.get("balls", 0),
            p.get("fours", 0),
            p.get("sixes", 0),
            p.get("strike_rate", 0),
            p.get("out_desc")
        ))

    cur.executemany(query, values)
    conn.commit()
    cur.close()
    conn.close()

In [26]:
matches = extract_matches(data)
matches = convert_dates(matches)

# insert matches first
insert_matches(matches)

# for each match → extract + insert players
for m in matches:
    match_id = m["match_id"]

    # ⚠️ here you must fetch scorecard API for each match
    players = extract_players(scorecard, match_id)

    insert_players(players)

C:\Users\hp\AppData\Local\Temp\ipykernel_20536\959014740.py:3: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  m["start_date"] = pd.to_datetime(m["start_date"], unit='ms') if m["start_date"] else None
C:\Users\hp\AppData\Local\Temp\ipykernel_20536\959014740.py:4: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  m["end_date"] = pd.to_datetime(m["end_date"], unit='ms') if m["end_date"] else None
